# 0. Prepare the Final Experiment

## 1. Imports and paths

outputs `notebooks/experiments/`.

In [2]:
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "final":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
elif PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from settings.final_config import (
    FINAL_SEEDS,
    FUTURE_KNOWN_COVARIATE_COLUMNS,
    GRID,
    INTERMITTENT_LABEL_PERIOD,
    PAST_COVARIATE_COLUMNS,
    RESULT_ROOT,
    SEARCH_SEED,
    SEARCH_TRAIN_END_IDX,
    STATIC_METADATA_COLUMNS,
    TEST_END_IDX,
    TEST_START_IDX,
    VALIDATION_END_IDX,
    VALIDATION_START_IDX,
)
from settings.final_data import (
    build_split_ids,
    fit_covariate_scaler,
    fit_metadata_encoders,
    load_existing_split,
    save_preprocessors,
)

RESULT_ROOT

WindowsPath('C:/Users/Francisco/Documents/Documentos-secundario/Visual-Studio-Code-Repository/CT5108_Capstone/Unified_forecasting/notebooks/experiments')

## 2. Create experiment folders

Create folders 
- split records, 
- preprocessors, 
- grid results, 
- checkpoints, 
- predictions,
- training histories

In [3]:
experiment_folders = {
    "splits": RESULT_ROOT / "splits",
    "preprocessors": RESULT_ROOT / "preprocessors",
    "grid_results": RESULT_ROOT / "grid_results",
    "checkpoints": RESULT_ROOT / "checkpoints",
    "predictions": RESULT_ROOT / "predictions",
    "training_history": RESULT_ROOT / "training_history",
}

for folder in experiment_folders.values():
    folder.mkdir(parents=True, exist_ok=True)

experiment_folders

{'splits': WindowsPath('C:/Users/Francisco/Documents/Documentos-secundario/Visual-Studio-Code-Repository/CT5108_Capstone/Unified_forecasting/notebooks/experiments/splits'),
 'preprocessors': WindowsPath('C:/Users/Francisco/Documents/Documentos-secundario/Visual-Studio-Code-Repository/CT5108_Capstone/Unified_forecasting/notebooks/experiments/preprocessors'),
 'grid_results': WindowsPath('C:/Users/Francisco/Documents/Documentos-secundario/Visual-Studio-Code-Repository/CT5108_Capstone/Unified_forecasting/notebooks/experiments/grid_results'),
 'checkpoints': WindowsPath('C:/Users/Francisco/Documents/Documentos-secundario/Visual-Studio-Code-Repository/CT5108_Capstone/Unified_forecasting/notebooks/experiments/checkpoints'),
 'predictions': WindowsPath('C:/Users/Francisco/Documents/Documentos-secundario/Visual-Studio-Code-Repository/CT5108_Capstone/Unified_forecasting/notebooks/experiments/predictions'),
 'training_history': WindowsPath('C:/Users/Francisco/Documents/Documentos-secundario/Visu

## 3. Record the existing split

Normal and intermittent series are used for training

Intermittent labels used the full history

In [ ]:
series_table = load_existing_split()
train_ids, split_ids = build_split_ids(series_table)

split_summary = (
    series_table["exclusive_split"]
    .value_counts()
    .rename_axis("subset")
    .reset_index(name="number_of_series")
)

all_combined_count = pd.DataFrame({
    "subset": ["all_combined"],
    "number_of_series": [len(split_ids["all_combined"])],
})
split_summary = pd.concat(
    [split_summary, all_combined_count],
    ignore_index=True,
)

# Save an exact copy for traceability
series_table.to_csv(
    experiment_folders["splits"] / "combined_exclusive_series_used.csv",
    index=False,
)
split_summary.to_csv(
    experiment_folders["splits"] / "existing_split_summary.csv",
    index=False,
)

print("Training series:", len(train_ids))
print("Intermittent label period:", INTERMITTENT_LABEL_PERIOD)
split_summary

Training series: 24696
Intermittent label period: d_1 to d_1913


,subset,number_of_series
0,intermittent,17096
1,normal,7600
2,cold_start_item,3050
3,cold_start_store,2744
4,all_combined,30490


## 4. Fit search preprocessors

Scaling uses only normal and intermittent series and only dates of the search training period

In [5]:
search_scaler, search_price_fallback = fit_covariate_scaler(
    series_ids=train_ids,
    covariate_columns=PAST_COVARIATE_COLUMNS,
    fit_end_idx=SEARCH_TRAIN_END_IDX,
)
search_metadata_encoders, search_cardinalities = fit_metadata_encoders(
    series_table=series_table,
    series_ids=train_ids,
    metadata_columns=STATIC_METADATA_COLUMNS,
)

save_preprocessors(
    output_dir=experiment_folders["preprocessors"],
    scaler=search_scaler,
    price_fallback=search_price_fallback,
    metadata_encoders=search_metadata_encoders,
    prefix="search",
)

print("Search price fallback:", search_price_fallback)
print("Search metadata cardinalities:", search_cardinalities)

Search price fallback: 3.47
Search metadata cardinalities: [4, 8, 4]


## 5. Fit final preprocessors

Adds the validation period but still uses only the training series

The final test period is not used

In [6]:
final_scaler, final_price_fallback = fit_covariate_scaler(
    series_ids=train_ids,
    covariate_columns=PAST_COVARIATE_COLUMNS,
    fit_end_idx=VALIDATION_END_IDX,
)
final_metadata_encoders, final_cardinalities = fit_metadata_encoders(
    series_table=series_table,
    series_ids=train_ids,
    metadata_columns=STATIC_METADATA_COLUMNS,
)

save_preprocessors(
    output_dir=experiment_folders["preprocessors"],
    scaler=final_scaler,
    price_fallback=final_price_fallback,
    metadata_encoders=final_metadata_encoders,
    prefix="final",
)

print("Final price fallback:", final_price_fallback)
print("Final metadata cardinalities:", final_cardinalities)

Final price fallback: 3.47
Final metadata cardinalities: [4, 8, 4]


## 6. Save the final settings

The file records the common periods, features, seeds, and grid used by every final notebook.

In [7]:
protocol = {
    "split_source": "data/m5/processed/splits/combined_exclusive_series.csv",
    "split_regenerated": False,
    "intermittent_label_period": INTERMITTENT_LABEL_PERIOD,
    "training_groups": ["normal", "intermittent"],
    "search_training_days": "d_1 to d_1857",
    "validation_days": "d_1858 to d_1885",
    "final_training_days": "d_1 to d_1885",
    "test_days": "d_1886 to d_1913",
    "search_train_end_idx": SEARCH_TRAIN_END_IDX,
    "validation_start_idx": VALIDATION_START_IDX,
    "validation_end_idx": VALIDATION_END_IDX,
    "test_start_idx": TEST_START_IDX,
    "test_end_idx": TEST_END_IDX,
    "past_covariates": PAST_COVARIATE_COLUMNS,
    "future_known_covariates": FUTURE_KNOWN_COVARIATE_COLUMNS,
    "static_metadata": STATIC_METADATA_COLUMNS,
    "search_seed": SEARCH_SEED,
    "final_seeds": FINAL_SEEDS,
    "grid": GRID,
    "selection_rule": "minimum aggregate validation WAPE",
}

with open(
    RESULT_ROOT / "final_experiment_protocol.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(protocol, file, indent=2)

protocol

{'split_source': 'data/m5/processed/splits/combined_exclusive_series.csv',
 'split_regenerated': False,
 'intermittent_label_period': 'd_1 to d_1913',
 'training_groups': ['normal', 'intermittent'],
 'search_training_days': 'd_1 to d_1857',
 'validation_days': 'd_1858 to d_1885',
 'final_training_days': 'd_1 to d_1885',
 'test_days': 'd_1886 to d_1913',
 'search_train_end_idx': 1856,
 'validation_start_idx': 1857,
 'validation_end_idx': 1884,
 'test_start_idx': 1885,
 'test_end_idx': 1912,
 'past_covariates': ['day_of_week',
  'is_weekend',
  'event_flag',
  'snap_flag',
  'sell_price'],
 'future_known_covariates': ['day_of_week',
  'is_weekend',
  'event_flag',
  'snap_flag'],
 'static_metadata': ['cat_id', 'dept_id', 'state_id'],
 'search_seed': 42,
 'final_seeds': [42, 300, 2026],
 'grid': {'seq_len': [28, 56, 112],
  'learning_rate': [0.001, 0.0003],
  'epochs': [50],
  'samples_per_epoch': [10000, 20000, 40000]},
 'selection_rule': 'minimum aggregate validation WAPE'}

Individual notebooks 01-06 perform the corresponding
- grid search (train)
- selects best configuration using WAPE (validation)
- trains using that configuration (train+validation)
- evaluates (final test)